# Phase 2 · Experiment 2A — Quantitative Characterisation

### Google Colab notebook (independent)

Establishes the quantitative **baseline** every later experiment compares against: the `mean_from_k` sink score computed as a Layer × Head matrix across all Phase 1 prompts. Output is **numerical artefacts** (tensors, profiles, summary stats). Per the Phase 2 brief, the Layer × Head matrix is saved as **data**, and qualitative attention figures from Phase 1 are **not** reproduced.

**Runtime:** CPU is fine — this notebook only reads Phase 1 attention.

**Prerequisites**
- Phase 1 must have been run with `USE_DRIVE=True`, so `attention_sink_data/` is in your Drive project folder.
- `phase2_utils.py` must be uploaded into the same Drive project folder (upload once; it persists).
- No model download needed.

This notebook is self-contained: it can be rerun on its own without executing the other experiments.

---

## 0. Setup

In [ ]:
# --- Colab environment setup -------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *['scipy', 'pyarrow']], check=True)

print('In Colab:', IN_COLAB)
print('Analysis-only notebook: no GPU required.')


In [ ]:
# --- Storage + phase2_utils bootstrap ---------------------------------------
# Point at the SAME Drive project folder Phase 1 used, so Phase 1's raw
# attention and this project's phase2_utils.py are both visible.
USE_DRIVE = True   # must match the Phase 1 setting

import sys
from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')
BASE.mkdir(parents=True, exist_ok=True)

DATA_ROOT    = str(BASE / 'attention_sink_data')          # Phase 1 output
RESULTS_ROOT = str(BASE / 'results' / 'phase2')           # Phase 2 output
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

# Locate phase2_utils.py (upload it into BASE once; it persists on Drive).
for c in [BASE, Path('/content'), Path('.')]:
    if (Path(c) / 'phase2_utils.py').exists():
        sys.path.insert(0, str(c)); break
try:
    import phase2_utils as U
    print('phase2_utils loaded from', U.__file__)
except ModuleNotFoundError:
    raise SystemExit('Place phase2_utils.py in ' + str(BASE) + ' (or /content) and re-run this cell.')

print('Phase 1 data :', DATA_ROOT)
print('Phase 2 out  :', RESULTS_ROOT)


## 1. Configuration

In [ ]:
from dataclasses import dataclass, asdict
import json, numpy as np, pandas as pd

@dataclass
class Config2A:
    k: int = 4                    # mean_from_k query-skip (Phase 1 primary setting)
    sink_threshold: float = 0.30
    seed: int = 20240517

cfg = Config2A()
U.set_reproducibility(cfg.seed)
EXP = U.experiment_dir(RESULTS_ROOT, 'experiment2A')
logger = U.get_logger('2A', log_file=str(EXP / 'run.log'))
logger.info('Config: %s', asdict(cfg))

## 2. Load Phase 1 metadata & check comparability

In [ ]:
meta = pd.read_csv(Path(DATA_ROOT) / 'metadata.csv')
assert (Path(DATA_ROOT) / 'attn').exists(), 'Phase 1 attention not found at ' + DATA_ROOT
# token 0 must be standardised for cross-prompt comparability (Phase 1 guarantee).
if meta['token0_id'].nunique() != 1:
    logger.warning('token0_id is not constant across prompts; baseline comparability is weakened.')
print('prompts:', len(meta), '| languages:', meta['language'].value_counts().to_dict())
print('layers x heads:', int(meta['n_layers'].iloc[0]), 'x', int(meta['n_heads'].iloc[0]))

## 3. Compute the baseline sink tensor  [P, L, H]

In [ ]:
tensor = U.compute_sink_tensor(DATA_ROOT, meta['prompt_id'].tolist(), k=cfg.k)
base = U.summarize_baseline(tensor, thr=cfg.sink_threshold)
print('tensor:', tensor.shape)
print(json.dumps(base['summary'], indent=2))

## 4. Save numerical artefacts (matrix saved as DATA, not a figure)

In [ ]:
U.save_baseline(EXP, tensor, base, meta['prompt_id'].tolist())
print('saved to', EXP)
print(sorted(p.name for p in EXP.iterdir() if p.is_file()))

## 5. Baseline reference plots (quantitative profiles only)

In [ ]:
U.plot_layer_profile({'all prompts': base['layer_profile']}, EXP / 'figures',
                     'layer_profile.png', stds={'all prompts': base['layer_profile_std']},
                     title='Baseline layer profile (mean_from_k)')
U.plot_head_profile({'all prompts': base['head_profile']}, EXP / 'figures',
                    'head_profile.png', title='Baseline head profile (mean_from_k)')

## Download results

In [ ]:
# --- Download this experiment's outputs -------------------------------------
import shutil
exp = Path(RESULTS_ROOT) / 'experiment2A'
zp = shutil.make_archive(str(Path('/content' if IN_COLAB else '.') / ('experiment2A_outputs')), 'zip', exp)
print('Bundled:', zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)
